# Реализовать с сравнить (на выбранной ранее среде) друг с другом и с обычным DQN следующие его модификации:
- DQN с Hard Target Update; 
- DQN с Soft Target Update;
- Double DQN.

# Libraries

In [1]:
from typing import Optional, Literal
import random
from warnings import warn

from tqdm.auto import tqdm
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Optimizer, Adam, AdamW

# Deep Q Network

## ReplayBuffer: memory allocation and management

In [2]:
class MemoryBlock: 
    def __init__(self, 
                 state: torch.FloatTensor | np.ndarray, 
                 action: torch.IntTensor | np.ndarray, 
                 reward: torch.FloatTensor | np.ndarray, 
                 next_state: torch.FloatTensor | np.ndarray, 
                 done: torch.IntTensor | np.ndarray,
                 device: str = None
                 ):
        self.state = torch.from_numpy(state) if isinstance(state, np.ndarray) else state
        self.action = torch.tensor(action) if isinstance(state, np.ndarray) else action
        self.reward = torch.tensor(reward) if isinstance(state, np.ndarray) else reward
        self.next_state = torch.from_numpy(next_state) if isinstance(state, np.ndarray) else next_state
        self.done = torch.tensor(done) if isinstance(state, np.ndarray) else done
        
        if device:
            self.to(device)
        self.__check_device()
        
    def to(self, device) -> None:
        self.state = self.state.to(device)
        self.action = self.action.to(device)
        self.reward = self.reward.to(device)
        self.next_state = self.next_state.to(device)
        self.done = self.done.to(device)

    def __str__(self) -> str:
        return (
            f"MemoryBlock(\n"
            f"state={self.state!r}\n"
            f"action={self.action!r}\n"
            f"reward={self.reward!r}\n"
            f"next_state={self.next_state!r}\n"
            f"done={self.done!r}\n)"
        )
    
    def __repr__(self) -> str:
        return (
            f"MemoryBlock("
            f"state={self.state!r}, "
            f"action={self.action!r}, "
            f"reward={self.reward!r}, "
            f"next_state={self.next_state!r}, "
            f"done={self.done!r})"
        )
        
    def __check_device(self) -> None:
        shared_device = {
            self.state.device, self.action.device, self.reward.device, 
            self.next_state.device, self.done.device
        }
        if len(shared_device) != 1:
            raise AssertionError(
                f"All element in memory block must be in the same device, but got: "
                f"{self.state.device=}, {self.action.device=}, {self.reward.device=}, "
                f"{self.next_state.device=}, {self.done.device}"
            )
        
class Memory:
    def __init__(self,
                 buffer_size: int, 
                 state_dim: int, 
                 device: Literal["cpu", "cuda"]
                 ):
        self.device = device
        self.buffer_size = buffer_size
        self.counter = 0
        self._states = torch.empty(self.buffer_size, state_dim, dtype=torch.float32, device=self.device)
        self._actions = torch.empty(self.buffer_size, 1, dtype=torch.int64, device=self.device)
        self._rewards = torch.empty(self.buffer_size, 1, dtype=torch.float32, device=self.device)
        self._next_states = torch.empty(self.buffer_size, state_dim, dtype=torch.float32, device=self.device)
        self._dones = torch.empty(self.buffer_size, 1, dtype=torch.int64, device=self.device)
        
    def __len__(self) -> int:
        return self.counter
    
    def __getitem__(self, index: int) -> MemoryBlock:
        return MemoryBlock(
            self._states[index], 
            self._actions[index], 
            self._rewards[index], 
            self._next_states[index], 
            self._dones[index]
        )
    
    def __setitem__(self, index: int, block: MemoryBlock) -> None:
        block.to(self.device)
        self._states[index] = block.state
        self._actions[index] = block.action
        self._rewards[index] = block.reward
        self._next_states[index] = block.next_state
        self._dones[index] = block.done
        self.counter += 1
    
    def write(self, block: MemoryBlock) -> None:
        index = self.counter % self.buffer_size
        self[index] = block


class ReplayBuffer(Memory):
    def __init__(self, 
                 buffer_size: int, 
                 batch_size: int, 
                 state_dim: int, 
                 device: Literal["cpu", "cuda"]
                 ):
        super().__init__(buffer_size, state_dim, device)
        self.batch_size = batch_size
    
    def sample(self) -> MemoryBlock:
        if len(self) >= self.batch_size:
            indices = torch.randint(
                0, min(len(self), self.buffer_size), 
                (self.batch_size,),
                dtype=torch.int64
            )
            return self[indices]
        warn(
            f"Can't sample MemoryBlock from ReplayBuffer "
            f"before len {len(self)} less then batch_size {self.batch_size}"
        )

## Aka Rainbow DQN Agent

In [3]:
class DQN(nn.Module):
    def __init__(self, space_dim: int, action_n: int):
        super().__init__()
        self.space_dim = space_dim
        self.action_n = action_n
        self.layers = nn.Sequential(
            nn.Linear(space_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_n)
        )
    
    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        return self.layers(x)


class AkaRainbowAgent:
    def __init__(self, 
                 state_dim: int, 
                 action_n: int,
                 buffer_size: int, 
                 batch_size: int,
                 optimizer: Optimizer,
                 lr: Optional[float] = 0.0001,
                 gamma: Optional[float] = 0.98, 
                 eps: Optional[float] = 1.0, 
                 eps_min: Optional[float] = 0.01, 
                 eps_decay: Optional[float] = 0.98, 
                 device: Optional[Literal["cpu", "cuda"]] = None,
                 tau: Optional[float] = 0.005,
                 update_target_each: Optional[int] = 0,
                 double: Optional[bool] = False
                 ):
        assert 1.0 >= tau >= 0., "Coefficient soft target update 'tau' must be between 0 and 1"

        if device is None:
            warn("No device was defined, device will be selected automatically")
            device = "cuda" if torch.cuda.is_available() else "cpu"
            
        self.device = device 
        self.main_dqn = DQN(state_dim, action_n).to(self.device)
        self.target_dqn = self._init_target_network()
        
        self.buffer = ReplayBuffer(buffer_size, batch_size, state_dim, device)
        self.criterion = nn.MSELoss()
        self.optimizer = optimizer(self.main_dqn.parameters(), lr=lr)
        self.gamma = gamma
        self.eps = eps
        self.eps_min = eps_min
        self.eps_decay = eps_decay
        
        self.double = double
        self.tau = tau
        self.update_target_each = update_target_each
        self.target_update_counter = 0
        self.overestimation_tracker = []
    
    def _init_target_network(self) -> nn.Module:
        target_dqn = DQN(self.main_dqn.space_dim, self.main_dqn.action_n).to(self.device)
        target_dqn.load_state_dict(self.main_dqn.state_dict())
        target_dqn.eval()
        return target_dqn
        
    def get_action(self, state: np.ndarray | torch.FloatTensor) -> int:
        if self.main_dqn.training and random.uniform(0, 1) < self.eps:
            action = np.random.randint(self.main_dqn.action_n)
        else:
            if isinstance(state, np.ndarray):
                state = torch.FloatTensor(state)
            state = state.to(self.device)
            with torch.no_grad():
                action = torch.argmax(self.main_dqn(state).cpu()).item()
        return action
        
    def fit(self, data: MemoryBlock) -> dict:
        self.buffer.write(data) 
        loss = None
       
        if shuffled_memory := self.buffer.sample(): 
            predicted_q_values = self.main_dqn.forward(shuffled_memory.state).gather(1, shuffled_memory.action)
            
            estimated_q_values = predicted_q_values.detach().cpu().numpy()
            actual_rewards = shuffled_memory.reward.cpu().numpy()
            overestimation = np.mean(estimated_q_values - actual_rewards)
            self.overestimation_tracker.append(overestimation)
            
            with torch.no_grad():
                if self.double:
                    next_state_actions = self.main_dqn.forward(shuffled_memory.next_state).max(1)[1].unsqueeze(1)
                    max_next_q_values = self.target_dqn(shuffled_memory.next_state).gather(1, next_state_actions)
                else:
                    max_next_q_values = self.target_dqn.forward(shuffled_memory.next_state).max(1)[0].unsqueeze(1)
                target_q_values = shuffled_memory.reward + (self.gamma * max_next_q_values * (1 - shuffled_memory.done))
            
            loss = self.criterion(predicted_q_values, target_q_values)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            if self.eps > self.eps_min:
                self.eps *= self.eps_decay
            self.target_update()
                
        return {
            "is_trained": True if loss else False,
            "loss": loss.detach().cpu().item() if loss else float("nan"),
            "eps": self.eps,
        }
    
    def target_update(self) -> None:
        if self.tau < 1:
            self.soft_target_update()
        elif self.target_update_counter % self.update_target_each == 0:
            self.hard_target_update()
        self.target_update_counter += 1
    
    def hard_target_update(self) -> None:
        self.target_dqn.load_state_dict(self.main_dqn.state_dict())
    
    def soft_target_update(self) -> None:
        compute_update = lambda mp, tp: self.tau * mp.data + tp.data * (1. - self.tau)
        main_params, target_params = self.main_dqn.parameters(), self.target_dqn.parameters() 
        for main_param, target_param in zip(main_params, target_params):            
            param = compute_update(main_param, target_param)
            target_param.data.copy_(param)

    def overestimation_report(self) -> str:
        if not self.overestimation_tracker:
            return "Empty Overestimation tracker"
        avg_overestimation = np.mean(self.overestimation_tracker)
        return f"Average Overestimation: {avg_overestimation}"

## Utility func

In [4]:
def dqn(
        env: gym.Env, 
        agent: AkaRainbowAgent,
        num_trajectories: int, 
        max_trajectory_len: int = 5_000,
        goal: int = 200
):
    total_rewards = []
    for i in tqdm(range(num_trajectories), desc="Initialize DQN training 🚀"):
        total_reward = 0
    
        state, info = env.reset()
        for _ in range(max_trajectory_len):
            action = agent.get_action(state)
            
            next_state, reward, done, truncated, info = env.step(action)
            total_reward += reward
            
            data = MemoryBlock(
                state, action, reward,
                next_state, done
            )
            agent.fit(data)
            state = next_state
    
            if done:
                break
    
        total_rewards.append(total_reward)
        if len(total_rewards) >= 100 and np.mean(total_rewards[-100:]) >= goal:
            print("Converged 🟩")
            break 
            
        overestimation_report = agent.overestimation_report()
        print(f"Total Reward: {total_reward} At {i} trajectory With {overestimation_report}")
    return total_rewards + [total_rewards[-1]] * (num_trajectories - len(total_rewards))


In [5]:
def seed(seed: int = 42) -> None:
    torch.random.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

# Setup

## ENV

In [6]:
env = gym.make("LunarLander-v2", render_mode=None)
print(env.observation_space.shape, "\n", env.action_space)

(8,) 
 Discrete(4)


## Constants

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ITERATIONS = 600
SEED = 45

/home/merci/code/PycharmProjects/deep-RL/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:128: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [8]:
seed(SEED)

# Train

## DQN vanila

In [9]:
dqn_agent = AkaRainbowAgent(
    state_dim=8,
    action_n=4,
    buffer_size=100_000,
    batch_size=1024,
    optimizer=Adam,
    lr=0.0001,
    device=DEVICE,
    double=False,
    update_target_each=1,
    tau=1
)

vanila_dqn_total_rewards = dqn(env, dqn_agent, num_trajectories=ITERATIONS)

Initialize DQN training 🚀:   0%|          | 0/600 [00:00<?, ?it/s]

/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 1 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 2 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 3 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 4 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 5 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 6 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 7 less then batch_size 1024


Total Reward: -98.18819018957252 At 0 trajectory With Empty Overestimation tracker
Total Reward: -139.04988069290278 At 1 trajectory With Empty Overestimation tracker
Total Reward: -67.09793985624984 At 2 trajectory With Empty Overestimation tracker
Total Reward: -152.26765415284677 At 3 trajectory With Empty Overestimation tracker
Total Reward: -143.7999162626881 At 4 trajectory With Empty Overestimation tracker
Total Reward: 0.9012115637426774 At 5 trajectory With Empty Overestimation tracker
Total Reward: -138.95472242970766 At 6 trajectory With Empty Overestimation tracker
Total Reward: -81.7411025762935 At 7 trajectory With Empty Overestimation tracker
Total Reward: -335.1335945615355 At 8 trajectory With Empty Overestimation tracker
Total Reward: -123.58449288099773 At 9 trajectory With Empty Overestimation tracker
Total Reward: -83.78014230599305 At 10 trajectory With Empty Overestimation tracker
Total Reward: -330.87311641075223 At 11 trajectory With Empty Overestimation tracke

## DDQN

In [14]:
dqn_agent = AkaRainbowAgent(
    state_dim=8,
    action_n=4,
    buffer_size=50_000,
    batch_size=1024,
    optimizer=Adam,
    lr=0.0005,
    device=DEVICE,
    double=True,
    update_target_each=1,
    tau=0.1
)


ddqn_total_rewards = dqn(env, dqn_agent, num_trajectories=ITERATIONS)

Initialize DQN training 🚀:   0%|          | 0/600 [00:00<?, ?it/s]

/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 1 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 2 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 3 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 4 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 5 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 6 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 7 less then batch_size 1024


Total Reward: -259.2016318419071 At 0 trajectory With Empty Overestimation tracker
Total Reward: -349.3920939559785 At 1 trajectory With Empty Overestimation tracker
Total Reward: -77.60603885203295 At 2 trajectory With Empty Overestimation tracker
Total Reward: -106.5830745875969 At 3 trajectory With Empty Overestimation tracker
Total Reward: -274.1079808821946 At 4 trajectory With Empty Overestimation tracker
Total Reward: -352.89997597663796 At 5 trajectory With Empty Overestimation tracker
Total Reward: -304.9836103392869 At 6 trajectory With Empty Overestimation tracker
Total Reward: -111.01346320264271 At 7 trajectory With Empty Overestimation tracker
Total Reward: -90.19796918021898 At 8 trajectory With Empty Overestimation tracker
Total Reward: -110.10137298609433 At 9 trajectory With Empty Overestimation tracker
Total Reward: -229.65928445105075 At 10 trajectory With Empty Overestimation tracker
Total Reward: -89.75111728385819 At 11 trajectory With Empty Overestimation tracke

## Soft Target Update

In [15]:
dqn_agent = AkaRainbowAgent(
    state_dim=8,
    action_n=4,
    buffer_size=100_000,
    batch_size=1024,
    optimizer=Adam,
    lr=0.0005,
    device=DEVICE,
    double=False,
    update_target_each=1,
    tau=0.1
)

dqn_soft_total_rewards = dqn(env, dqn_agent, num_trajectories=ITERATIONS)

Initialize DQN training 🚀:   0%|          | 0/600 [00:00<?, ?it/s]

Total Reward: -131.6877031210538 At 0 trajectory With Empty Overestimation tracker
Total Reward: -250.19190208649903 At 1 trajectory With Empty Overestimation tracker
Total Reward: -154.99529277409687 At 2 trajectory With Empty Overestimation tracker
Total Reward: -85.9992761143635 At 3 trajectory With Empty Overestimation tracker
Total Reward: -131.86311665466502 At 4 trajectory With Empty Overestimation tracker
Total Reward: -104.7207154195155 At 5 trajectory With Empty Overestimation tracker
Total Reward: -148.3257308942144 At 6 trajectory With Empty Overestimation tracker
Total Reward: -92.95493640530344 At 7 trajectory With Empty Overestimation tracker
Total Reward: -325.056543799364 At 8 trajectory With Empty Overestimation tracker
Total Reward: -342.62760091671635 At 9 trajectory With Empty Overestimation tracker
Total Reward: -65.09697476936446 At 10 trajectory With Empty Overestimation tracker
Total Reward: -159.77202779747512 At 11 trajectory With Average Overestimation: 1.71

/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 1 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 2 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 3 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 4 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 5 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 6 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 7 less then batch_size 1024


Total Reward: -587.1667738961775 At 12 trajectory With Average Overestimation: -4.1144022941589355
Total Reward: -242.86387865833993 At 13 trajectory With Average Overestimation: -7.391052722930908
Total Reward: -733.6660906545378 At 14 trajectory With Average Overestimation: 14.15112590789795
Total Reward: -766.0505197702817 At 15 trajectory With Average Overestimation: 12.86255931854248
Total Reward: -757.5757089997012 At 16 trajectory With Average Overestimation: 9.742289543151855
Total Reward: -740.3844880877364 At 17 trajectory With Average Overestimation: 7.502499580383301
Total Reward: -746.1230510207024 At 18 trajectory With Average Overestimation: 5.899415969848633
Total Reward: -712.388612478459 At 19 trajectory With Average Overestimation: 5.038778781890869
Total Reward: -176.69665987718486 At 20 trajectory With Average Overestimation: 5.025435924530029
Total Reward: -222.35126188867872 At 21 trajectory With Average Overestimation: 5.01719856262207
Total Reward: -221.8992578

## Hard Target Update

In [12]:
dqn_agent = AkaRainbowAgent(
    state_dim=8,
    action_n=4,
    buffer_size=100_000,
    batch_size=1024,
    optimizer=Adam,
    lr=0.0005,
    device=DEVICE,
    double=False,
    update_target_each=10,
    tau=1
)

dqn_hard_total_rewards = dqn(env, dqn_agent, num_trajectories=ITERATIONS)

Initialize DQN training 🚀:   0%|          | 0/600 [00:00<?, ?it/s]

Total Reward: -112.64076838259176 At 0 trajectory With Empty Overestimation tracker
Total Reward: -112.11866791335434 At 1 trajectory With Empty Overestimation tracker
Total Reward: -144.4728900541805 At 2 trajectory With Empty Overestimation tracker
Total Reward: -188.9610183745939 At 3 trajectory With Empty Overestimation tracker
Total Reward: -327.3844368916406 At 4 trajectory With Empty Overestimation tracker
Total Reward: -95.21925759065047 At 5 trajectory With Empty Overestimation tracker


/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 1 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 2 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 3 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 4 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 5 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 6 less then batch_size 1024
  warn(
/tmp/ipykernel_3397962/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 7 less then batch_size 1024


Total Reward: 172.68444911427503 At 6 trajectory With Average Overestimation: -1.4240331649780273
Total Reward: -46.51537582141682 At 7 trajectory With Average Overestimation: -0.983190655708313
Total Reward: -403.00684228262946 At 8 trajectory With Average Overestimation: 7.204333305358887
Total Reward: -486.3658624735961 At 9 trajectory With Average Overestimation: 9.473427772521973
Total Reward: -292.60909832159473 At 10 trajectory With Average Overestimation: 11.884214401245117
Total Reward: -292.5361379560019 At 11 trajectory With Average Overestimation: 14.173084259033203
Total Reward: -66.66880907511083 At 12 trajectory With Average Overestimation: 17.378032684326172
Total Reward: -302.8023195968841 At 13 trajectory With Average Overestimation: 20.55031967163086
Total Reward: 43.49177072281552 At 14 trajectory With Average Overestimation: 22.30332374572754
Total Reward: -107.0593481395113 At 15 trajectory With Average Overestimation: 23.534526824951172
Total Reward: -112.5612884